# Steiner Tree Assignment

TEAM MEMBERS:

Kanishk Choudary 231EC223

Rishadd Ranjith 231EC241

Rohit L 231EC248

Q1 Complete

In [ ]:
import re, os, time, tracemalloc, random
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from sklearn.cluster import KMeans

# Parse input file
content = open(os.path.join(os.getcwd(), 'test_data_assignment_2.txt')).read()
raw     = list(map(int, re.findall(r'-?\d+', content)))
n       = raw[0]
coords  = [(raw[1 + 2*i], raw[2 + 2*i]) for i in range(n)]
print(f'n={n}, coords={coords}')

# Helper functions
R, C = 1.0, 1e-15 #resistance and Capacitance given

def seg_len(p1, p2): #segment length
    return abs(p1[0]-p2[0]) + abs(p1[1]-p2[1])

def elmore(lengths): #elmore delay calculation
    total, delay, cum = sum(lengths), 0.0, 0.0
    for L in lengths:
        delay += (R*L) * (C*(total-cum))
        cum   += L
    return delay

def bfs_path(src, dst, edges): #bfs path exploration
    g = defaultdict(list)
    for p1,p2,L in edges:
        g[p1].append((p2,L)); g[p2].append((p1,L))
    q, visited = deque([(src,[])]), set()
    while q:
        node, path = q.popleft()
        if node == dst: return path
        if node in visited: continue
        visited.add(node)
        for nb,L in g[node]:
            if nb not in visited: q.append((nb, path+[L]))
    return None

def all_delays(src, edges): #all elmore delays for each path
    g = defaultdict(list)
    for p1,p2,L in edges:
        g[p1].append((p2,L)); g[p2].append((p1,L))
    result, q, visited = {}, deque([(src,[])]), set()
    while q:
        node, path = q.popleft()
        if node in visited: continue
        visited.add(node)
        result[node] = elmore(path)
        for nb,L in g[node]:
            if nb not in visited: q.append((nb, path+[L]))
    return result

tracemalloc.start()
t0 = time.perf_counter()

# The trunk is a horizontal wire at the median y-coordinate,
# Median minimises Σ |y_i - trunk_y|, i.e. total vertical (branch) wire.
xs, ys   = [p[0] for p in coords], [p[1] for p in coords]
trunk_y  = int(np.median(ys))          # median y  →  optimal horizontal trunk
trunk_x1 = min(xs)
trunk_x2 = max(xs)
clk      = (trunk_x1, trunk_y)         # CLK at leftmost trunk point

print(f'Trunk: {{{trunk_x1}, {trunk_y}, {trunk_x2}, {trunk_y}}}')

# Horizontal trunk segments between consecutive unique x-positions
trunk_xs    = sorted(set(xs))
trunk_nodes = [(x, trunk_y) for x in trunk_xs]
edges       = []
for i in range(len(trunk_nodes) - 1):
    p1, p2 = trunk_nodes[i], trunk_nodes[i + 1]
    L = abs(p2[0] - p1[0])             # purely horizontal → |Δx|
    edges.append((p1, p2, L))

# Vertical branch from each input node straight to the trunk
# Manhattan rule: go vertically (|Δy|) since x is already shared.
steiner_pts = set()
for p in coords:
    cp = (p[0], trunk_y)               # foot of perpendicular on trunk
    L  = abs(p[1] - trunk_y)           # purely vertical → |Δy|
    if L > 0:
        edges.append((cp, p, L))
    if cp not in coords:
        steiner_pts.add(cp)

total_wire = sum(e[2] for e in edges)
print(f'Total wire length: {total_wire} units')

delays = {}
print('\nElmore Delays from CLK:')
print(f'  {"Node":<8} {"Coords":<14} {"Delay (fs)":>12}')
print('  ' + '─' * 38)
for i, p in enumerate(coords):
    path      = bfs_path(clk, p, edges)
    d         = elmore(path)
    delays[i] = d
    print(f'  Node {i+1:<3} {str(p):<14} {d * 1e15:>12.2f} fs')

crit = max(delays, key=lambda k: delays[k])
print(f'\nCritical net: Node {crit+1} {coords[crit]}  →  {delays[crit]*1e15:.2f} fs')

t1 = time.perf_counter()
mc, mp = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'\nTime  : {(t1 - t0)*1000:.3f} ms')
print(f'Memory: {mp / 1024:.2f} KB peak')

# Plot
fig, ax = plt.subplots(figsize=(10,7))

for p1,p2,L in edges:
    ax.plot([p1[0],p2[0]], [p1[1],p2[1]], 'b-')
    mx,my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
    d = elmore(bfs_path(clk,p2,edges) or [L])
    ax.text(mx, my+0.2, f'L={L}\nD={d*1e15:.1f}', fontsize=6, ha='center')

# Steiner points
for p in coords:
    cp = (p[0], trunk_y)
    if cp not in coords:
        ax.plot(*cp, 'o', color='orange', markersize=6)

# Input nodes
for i,p in enumerate(coords):
    color = 'red' if i == crit else 'green'
    ax.plot(*p, 'o', color=color, markersize=8)
    ax.text(p[0]+0.2, p[1]+0.2, f'N{i+1}', fontsize=7)

ax.plot(*clk, '*', color='purple', markersize=14, label='CLK')
ax.legend(['CLK'])
ax.set_title('Q1: Steiner Tree')
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.grid(True)
plt.tight_layout(); plt.show()


Q2 Complete

In [ ]:
import re, os, time, tracemalloc, random
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from sklearn.cluster import KMeans

# Parse input file
content = open(os.path.join(os.getcwd(), 'test_data_assignment_2.txt')).read()
raw     = list(map(int, re.findall(r'-?\d+', content)))
n       = raw[0]
coords  = [(raw[1 + 2*i], raw[2 + 2*i]) for i in range(n)]
print(f'n={n}, coords={coords}')

# Helper functions
R, C = 1.0, 1e-15 #resistance and Capacitance given

def seg_len(p1, p2): #segment length
    return abs(p1[0]-p2[0]) + abs(p1[1]-p2[1])

def elmore(lengths): #elmore delay calculation
    total, delay, cum = sum(lengths), 0.0, 0.0
    for L in lengths:
        delay += (R*L) * (C*(total-cum))
        cum   += L
    return delay

def bfs_path(src, dst, edges): #bfs path exploration
    g = defaultdict(list)
    for p1,p2,L in edges:
        g[p1].append((p2,L)); g[p2].append((p1,L))
    q, visited = deque([(src,[])]), set()
    while q:
        node, path = q.popleft()
        if node == dst: return path
        if node in visited: continue
        visited.add(node)
        for nb,L in g[node]:
            if nb not in visited: q.append((nb, path+[L]))
    return None

def all_delays(src, edges): #all elmore delays for each path
    g = defaultdict(list)
    for p1,p2,L in edges:
        g[p1].append((p2,L)); g[p2].append((p1,L))
    result, q, visited = {}, deque([(src,[])]), set()
    while q:
        node, path = q.popleft()
        if node in visited: continue
        visited.add(node)
        result[node] = elmore(path)
        for nb,L in g[node]:
            if nb not in visited: q.append((nb, path+[L]))
    return result

xs, ys   = [p[0] for p in coords], [p[1] for p in coords]
trunk_y  = int(np.median(ys))
trunk_x1, trunk_x2 = min(xs), max(xs)

trunk_xs    = sorted(set(xs))
trunk_nodes = [(x, trunk_y) for x in trunk_xs]

edges = []
for i in range(len(trunk_nodes) - 1):
    p1, p2 = trunk_nodes[i], trunk_nodes[i + 1]
    edges.append((p1, p2, abs(p2[0] - p1[0])))

steiner_pts = set()
for p in coords:
    cp = (p[0], trunk_y)
    L  = abs(p[1] - trunk_y)
    if L > 0:
        edges.append((cp, p, L))
    if cp not in coords:
        steiner_pts.add(cp)

all_tree_nodes = list(set(trunk_nodes) | set(coords) | steiner_pts)
total_wire     = sum(e[2] for e in edges)
print(f'Trunk: {{{trunk_x1}, {trunk_y}, {trunk_x2}, {trunk_y}}}')
print(f'Total wire length: {total_wire} units')

tracemalloc.start(); t0 = time.perf_counter()

best_skew, best_clk, best_delays = float('inf'), None, None
tree_pts = trunk_nodes + list(coords)

# Scan Hanan grid + fine trunk sweep
candidates = [(hx,hy) for hx in sorted(set(xs)) for hy in sorted(set(ys))]
candidates += [(round(tx,2), trunk_y) for tx in np.linspace(trunk_x1, trunk_x2, 300)]

for cand in candidates:
    if cand in coords: continue
    nearest = min(tree_pts, key=lambda p: seg_len(p,cand))
    d_map   = all_delays(nearest, edges)
    nd      = [d_map.get(p, float('inf')) for p in coords]
    if any(v == float('inf') for v in nd): continue
    skew = max(nd) - min(nd)
    if skew < best_skew:
        best_skew  = skew
        best_clk   = cand
        best_delays = {i: d_map[coords[i]] for i in range(n)}

print(f'Optimal CLK location: {{{best_clk[0]}, {best_clk[1]}}}')
print(f'Minimum skew: {best_skew*1e15:.4f} fF.Ohm')
print('\nDelays (fF.Ohm):')
for i,p in enumerate(coords):
    print(f'  Node {i+1} {p}: {best_delays[i]*1e15:.2f}')

t1=time.perf_counter(); mc,mp=tracemalloc.get_traced_memory(); tracemalloc.stop()
print(f'\nTime: {(t1-t0)*1000:.3f} ms | Memory: {mp/1024:.2f} KB peak')


# Plot
fig, ax = plt.subplots(figsize=(10,7))

for p1,p2,L in edges:
    ax.plot([p1[0],p2[0]], [p1[1],p2[1]], 'b-')
    mx,my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
    ax.text(mx, my+0.2, f'L={L}', fontsize=6, ha='center')

# Zero-delay wire from new CLK to nearest trunk node
nearest_clk = min(trunk_nodes, key=lambda p: seg_len(p, best_clk))
ax.plot([best_clk[0], nearest_clk[0]], [best_clk[1], nearest_clk[1]], 'm--')

for p in coords:
    cp = (p[0], trunk_y)
    if cp not in coords:
        ax.plot(*cp, 'o', color='orange', markersize=6)

for i,p in enumerate(coords):
    ax.plot(*p, 'go', markersize=8)
    ax.text(p[0]+0.2, p[1]+0.2, f'N{i+1}\n{best_delays[i]*1e15:.1f}fΩ', fontsize=6)

ax.plot(*best_clk, '*', color='purple', markersize=14)
ax.text(best_clk[0]+0.2, best_clk[1]-0.8, f'CLK*\n{best_clk}', fontsize=8, color='purple')

ax.set_title(f'Q2: Optimal CLK @ {best_clk} | Skew={best_skew*1e15:.2f} fF.Ohm')
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.grid(True)
plt.tight_layout(); plt.show()

tracemalloc.start()
t0 = time.perf_counter()

def evaluate_skew(src, edges):
    d_map = all_delays(src, edges)
    nd    = [d_map.get(p, None) for p in coords]
    if any(v is None for v in nd):
        return float('inf'), {}
    return max(nd) - min(nd), {i: nd[i] for i in range(n)}

# Build candidate set: Hanan grid + dense interior-of-edge sweep
hanan_xs = sorted(set(xs))
hanan_ys = sorted(set(ys))
candidates = set()

# Hanan grid points
for hx in hanan_xs:
    for hy in hanan_ys:
        candidates.add((hx, hy))

# Interior of each edge at 0.5-unit steps
for p1, p2, L in edges:
    steps = max(2, int(L / 0.5))
    for k in range(steps + 1):
        t  = k / steps
        cx = p1[0] + t * (p2[0] - p1[0])
        cy = p1[1] + t * (p2[1] - p1[1])
        candidates.add((round(cx, 2), round(cy, 2)))

# Remove actual input nodes (CLK must be at a *new* point)
candidates -= set(coords)

best_skew    = float('inf')
best_clk     = None
best_delays  = None
best_inject  = None   # nearest tree-node used as injection point

for cand in candidates:
    # Find the nearest tree node — zero-delay wire connects CLK to here
    inject = min(all_tree_nodes, key=lambda p: seg_len(p, cand))
    skew, d_map = evaluate_skew(inject, edges)
    if skew < best_skew:
        best_skew   = skew
        best_clk    = cand
        best_delays = d_map
        best_inject = inject

print(f'\nOptimal CLK location : {{{best_clk[0]}, {best_clk[1]}}}')
print(f'Injection point (tree): {best_inject}')
print(f'Minimum clock skew   : {best_skew*1e15:.4f} fs')

print('\nNode delays from optimal CLK:')
print(f'  {"Node":<8} {"Coords":<14} {"Delay (fs)":>12}')
print('  ' + '─' * 38)
for i, p in enumerate(coords):
    print(f'  Node {i+1:<3} {str(p):<14} {best_delays[i]*1e15:>12.2f} fs')

crit = max(best_delays, key=lambda k: best_delays[k])
print(f'\nCritical net: Node {crit+1} {coords[crit]}  →  {best_delays[crit]*1e15:.2f} fs')

t1 = time.perf_counter()
mc, mp = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'\nExecution time : {(t1-t0)*1000:.3f} ms')
print(f'Peak memory    : {mp/1024:.2f} KB')




Q3 Complete

In [ ]:
import re, os, time, tracemalloc, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from collections import defaultdict, deque
from sklearn.cluster import KMeans

# Parse input file
content = open(os.path.join(os.getcwd(), 'test_data_assignment_2.txt')).read()
raw     = list(map(int, re.findall(r'-?\d+', content)))
n       = raw[0]
coords  = [(raw[1 + 2*i], raw[2 + 2*i]) for i in range(n)]
print(f'n={n}, coords={coords}')

# Helper functions
R, C = 1.0, 1e-15 #resistance and Capacitance given

def seg_len(p1, p2): #segment length
    return abs(p1[0]-p2[0]) + abs(p1[1]-p2[1])

def elmore(lengths): #elmore delay calculation
    total, delay, cum = sum(lengths), 0.0, 0.0
    for L in lengths:
        delay += (R*L) * (C*(total-cum))
        cum   += L
    return delay

def bfs_path(src, dst, edges): #bfs path exploration
    g = defaultdict(list)
    for p1,p2,L in edges:
        g[p1].append((p2,L)); g[p2].append((p1,L))
    q, visited = deque([(src,[])]), set()
    while q:
        node, path = q.popleft()
        if node == dst: return path
        if node in visited: continue
        visited.add(node)
        for nb,L in g[node]:
            if nb not in visited: q.append((nb, path+[L]))
    return None

def all_delays(src, edges): #all elmore delays for each path
    g = defaultdict(list)
    for p1,p2,L in edges:
        g[p1].append((p2,L)); g[p2].append((p1,L))
    result, q, visited = {}, deque([(src,[])]), set()
    while q:
        node, path = q.popleft()
        if node in visited: continue
        visited.add(node)
        result[node] = elmore(path)
        for nb,L in g[node]:
            if nb not in visited: q.append((nb, path+[L]))
    return result

tracemalloc.start(); t0 = time.perf_counter()

try:
    random.seed(42); np.random.seed(42)
    rand_pts = [(random.randint(0,40), random.randint(0,40)) for _ in range(30)]
    if not rand_pts:
        raise ValueError("Failed to generate random points.")

    unique_pts = sorted(list(set(rand_pts)))
    num_unique = len(unique_pts)

    # Edge Case: Fewer unique points than expected clusters
    max_possible_k = min(8, num_unique)
    if max_possible_k < 1:
        raise ValueError("Insufficient unique points for clustering.")

    X = np.array(rand_pts)
    # Elbow method with error handling
    k_range = range(2, max_possible_k + 1) if max_possible_k >= 2 else [1]
    inertias = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
        inertias.append(km.inertia_)

    if len(inertias) > 2:
        best_k = int(list(k_range)[np.argmax(-np.diff(np.diff(inertias))) + 1])
    else:
        best_k = k_range[-1]

    print(f'Generated {len(rand_pts)} points. Optimal k: {best_k}')

    # Perform final clustering
    kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X)
    labels = kmeans.labels_

    refined_edges_q3, refined_clk_locs = [], []

    for cid in range(best_k):
        pts = [rand_pts[i] for i in range(len(rand_pts)) if labels[i] == cid]
        if len(pts) < 2: continue # Cannot form a trunk with < 2 points

        cxs, cys = [p[0] for p in pts], [p[1] for p in pts]
        ty = sorted(cys)[len(cys)//2]
        tx1, tx2 = min(cxs), max(cxs)
        tnodes = [(x, ty) for x in sorted(set(cxs))]

        cedges = []
        # Horizontal Trunk
        for i in range(len(tnodes)-1):
            L = seg_len(tnodes[i], tnodes[i+1])
            if L >= 0: cedges.append((tnodes[i], tnodes[i+1], L))
        # Vertical Stubs to points
        for p in pts:
            cp = (p[0], ty)
            L = seg_len(p, cp)
            if L > 0: cedges.append((cp, p, L))

        # Cluster-local optimal CLK placement search
        bsk, bcl = float('inf'), (tx1, ty)
        search_space = np.linspace(tx1, tx2, 50)
        for tx in search_space:
            cand = (round(float(tx), 2), ty)
            nearest = min(tnodes, key=lambda p: abs(p[0]-cand[0]))
            dm = all_delays(nearest, cedges)
            nd = [dm.get(p, float('inf')) for p in pts]
            if all(v < float('inf') for v in nd):
                sk = max(nd) - min(nd)
                if sk < bsk: bsk, bcl = sk, cand

        refined_edges_q3.append((cedges, cid))
        refined_clk_locs.append((bcl, cid))

    # Global Connectivity
    if refined_clk_locs:
        cluster_clks = [loc for loc, cid in refined_clk_locs]
        global_root = (np.median([p[0] for p in cluster_clks]), np.median([p[1] for p in cluster_clks]))
        global_trunk_y = global_root[1]
        global_edges = []
        proj_nodes = sorted(list(set([(clk[0], global_trunk_y) for clk in cluster_clks])))

        for i in range(len(proj_nodes)-1):
            global_edges.append((proj_nodes[i], proj_nodes[i+1], seg_len(proj_nodes[i], proj_nodes[i+1])))
        for clk in cluster_clks:
            proj = (clk[0], global_trunk_y)
            L = seg_len(clk, proj)
            if L > 0: global_edges.append((proj, clk, L))
        print("Global connectivity established successfully.")

except Exception as e:
    print(f"Robustness Error: {e}")
finally:
    t1 = time.perf_counter(); mc, mp = tracemalloc.get_traced_memory(); tracemalloc.stop()
    print(f'Time: {(t1-t0)*1000:.2f} ms | Peak Memory: {mp/1024:.2f} KB')

unique_pts = sorted(list(set(rand_pts)))
num_unique = len(unique_pts)
max_k = min(8, num_unique)
k_range = range(2, max_k + 1) if max_k >= 2 else [1]

inertias = []
if len(k_range) > 1:
    X_unique = np.array(unique_pts)
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_unique)
        inertias.append(km.inertia_)
    best_k = int(list(k_range)[np.argmax(-np.diff(np.diff(inertias))) + 1]) if len(inertias) > 2 else k_range[-1]
else:
    best_k = 1

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(np.array(rand_pts))
labels = kmeans_final.labels_

refined_edges_q3, refined_clk_locs = [], []

for cid in range(best_k):
    pts = [rand_pts[i] for i in range(len(rand_pts)) if labels[i] == cid]
    if len(pts) < 2: continue # Safety: skip single-point clusters

    cxs, cys = [p[0] for p in pts], [p[1] for p in pts]
    ty = sorted(cys)[len(cys)//2]
    tx1, tx2 = min(cxs), max(cxs)
    tnodes = [(x, ty) for x in sorted(set(cxs))]
    cedges = []
    for i in range(len(tnodes)-1):
        cedges.append((tnodes[i], tnodes[i+1], seg_len(tnodes[i], tnodes[i+1])))
    for p in pts:
        cp = (p[0], ty)
        if (L := seg_len(p, cp)) > 0: cedges.append((cp, p, L))

    # Cluster-local optimal CLK placement
    bsk, bcl = float('inf'), (tx1, ty)
    for tx in np.linspace(tx1, tx2, 100):
        cand = (round(float(tx), 2), ty)
        nearest = min(tnodes, key=lambda p: abs(p[0]-cand[0]))
        dm = all_delays(nearest, cedges)
        nd = [dm.get(p, float('inf')) for p in pts]
        if all(v < float('inf') for v in nd) and (sk := max(nd)-min(nd)) < bsk:
            bsk, bcl = sk, cand

    refined_edges_q3.append((cedges, cid))
    refined_clk_locs.append((bcl, cid))

# Global Connectivity Multi-Trunk Logic
cluster_clks = [loc for loc, cid in refined_clk_locs]
global_root = (sorted([p[0] for p in cluster_clks])[len(cluster_clks)//2], sorted([p[1] for p in cluster_clks])[len(cluster_clks)//2])
global_trunk_y = global_root[1]
global_edges = []
proj_nodes = sorted(list(set([(clk[0], global_trunk_y) for clk in cluster_clks])))
for i in range(len(proj_nodes)-1):
    global_edges.append((proj_nodes[i], proj_nodes[i+1], seg_len(proj_nodes[i], proj_nodes[i+1])))
for clk in cluster_clks:
    proj = (clk[0], global_trunk_y)
    if (L := seg_len(clk, proj)) > 0: global_edges.append((proj, clk, L))

# --- 3. Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].plot(range(2, max_k + 1), inertias, 'o-')
axes[0].axvline(best_k, color='red', ls='--', label=f'k={best_k}')
axes[0].set_title('Elbow Method'); axes[0].grid(True)

ax = axes[1]
cm = plt.cm.tab10(np.linspace(0, 1, best_k))
for cedges, cid in refined_edges_q3:
    for p1, p2, L in cedges: ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=cm[cid], alpha=0.6)
for i, (p1, p2, L) in enumerate(global_edges):
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='purple', ls='--', lw=1.5, label='Global Trunk/Stubs' if i==0 else '')
for i, p in enumerate(rand_pts): ax.plot(*p, 'o', color=cm[labels[i]], ms=5, mec='k')
for loc, cid in refined_clk_locs: ax.plot(*loc, '*', color='darkviolet', ms=10)
ax.plot(*global_root, 'D', color='red', ms=12, label='Global Root')

ax.legend(handles=[mlines.Line2D([],[],color='teal',label='Local Steiner'), mlines.Line2D([],[],color='purple',ls='--',label='Global Connectivity'), mlines.Line2D([],[],marker='*',color='darkviolet',ls='',label='Cluster CLKs'), mlines.Line2D([],[],marker='D',color='red',ls='',label='Global Root')], loc='upper right')
ax.set_title(f'Q3: Hierarchical Multi-Trunk Clock Tree (k={best_k})'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

